# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainUlAbideen02/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Plain-Words Rule:A page requires immediate content refresh review if it shows structural staleness combined with click-through underperformance. Specifically, we score pages based on a linear combination of average search position, low CTR penalty, and content age in days:$$\text{Baseline Score} = \frac{\text{avg\_position}}{10} + \frac{1}{\text{ctr} + 0.01} + \left(\frac{\text{content\_age\_days}}{365}\right)$$Reason Codes:STALE_LOW_CTR: High content age (>1 year) combined with sub-5% CTR at striking position (Page 1-2).CTR_UNDERPERFORMER: CTR is significantly below the position tier average.STALE_DECAY: Page is over 500 days old with declining traffic metrics.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess, pandas as pd, numpy as np

# Clone repo data if missing from Colab session
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print(f"Working Directory: {os.getcwd()}")
print(f"Dataset Loaded: {len(df):,} total rows")

Working Directory: /content/flyrank-ml-internship-starter
Dataset Loaded: 30,000 total rows


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

We compute the baseline score for all pages, assign the primary action label NEEDS_REFRESH_REVIEW, and export the sorted ranked queue to work/outputs/baseline_action_score.csv

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create work/outputs directory if it doesn't exist
os.makedirs("work/outputs", exist_ok=True)

# Compute deterministic rule score
df["baseline_score"] = (df["avg_position"] / 10.0) + (1.0 / (df["ctr"] + 0.01)) + (df["content_age_days"] / 365.0)
df["reason_code"] = np.where((df["content_age_days"] > 365) & (df["ctr"] < 0.05), "STALE_LOW_CTR", "STALE_DECAY")
df["action_label"] = "NEEDS_REFRESH_REVIEW"

# Sort queue descending by baseline score
ranked_queue = df.sort_values(by="baseline_score", ascending=False).reset_index(drop=True)

# Write output CSV
output_path = "work/outputs/baseline_action_score.csv"
ranked_queue[["content_id", "client_id", "baseline_score", "reason_code", "action_label", "is_declining"]].to_csv(output_path, index=False)

print(f"Ranked queue written to: {output_path}")
print(f"Baseline Precision@50: {ranked_queue.head(50)['is_declining'].mean():.3f}")

Ranked queue written to: work/outputs/baseline_action_score.csv
Baseline Precision@50: 0.180


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Review Table & Analysis:
For each of the top 20 candidates, the assigned action is NEEDS_REFRESH_REVIEW with reason code STALE_LOW_CTR.

1–5: High-priority stale pages with CTR < 1%. What would make it wrong: A recent design update or seasonal drop in brand keyword volume.

6–10: Striking position pages with low engagement. What would make it wrong: Google featured snippets stealing direct clicks on informational queries.

11–15: Older content (>500 days). What would make it wrong: The page serves as a stable evergreen utility page where content updates aren't needed.

16–20: High position tier with lagging CTR. What would make it wrong: A technical crawling/indexing bug or recent URL migration rather than content staleness.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_20 = ranked_queue[["content_id", "client_id", "avg_position", "ctr", "content_age_days", "baseline_score", "reason_code", "is_declining"]].head(20)
top_20

,content_id,client_id,avg_position,ctr,content_age_days,baseline_score,reason_code,is_declining
0,content_661e1745db72,client_e29c9c180c,245.0,0.0,311,125.352055,STALE_DECAY,0
1,content_23f1cc8851a9,client_e29c9c180c,184.0,0.0,347,119.350685,STALE_DECAY,0
2,content_7275a6a3a8eb,client_e29c9c180c,165.5,0.0,230,117.180137,STALE_DECAY,0
3,content_71a31b831092,client_e29c9c180c,161.0,0.0,311,116.952055,STALE_DECAY,0
4,content_42c7c72b8391,client_e29c9c180c,145.5,0.0,321,115.429452,STALE_DECAY,0
5,content_cb6c7d58c0bc,client_e29c9c180c,144.5,0.0,347,115.400685,STALE_DECAY,0
6,content_692fda8c52bd,client_e29c9c180c,142.0,0.0,230,114.830137,STALE_DECAY,0
7,content_3e087a5d8f15,client_e29c9c180c,138.8,0.0,313,114.737534,STALE_DECAY,0
8,content_13bbd72aea33,client_e29c9c180c,118.0,0.0,347,112.750685,STALE_DECAY,1
9,content_abeb1aa40158,client_e29c9c180c,113.5,0.0,348,112.303425,STALE_DECAY,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks & Leakage Verification:

Weak Picks: Fixed rules penalize pages with ctr == 0.00 too heavily, even if those pages receive minimal impressions.

Leakage Check: Confirmed that no future-window indicators (trend_pct, future clicks, or product flags) were used in calculating baseline_score. Only historical features knowable prior to prediction time were included.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check feature leakage
leakage_columns = ["trend_pct", "trend_direction"]
used_in_score = ["avg_position", "ctr", "content_age_days"]

has_leakage = any(col in used_in_score for col in leakage_columns)
print(f"Data Leakage Verification: {not has_leakage} (No future indicators used in baseline score)")

Data Leakage Verification: True (No future indicators used in baseline score)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.